<a href="https://colab.research.google.com/github/B-Pearlraj/EQ_ANALYSIS/blob/main/EQ_ANALYSIS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pandas

In [ ]:
import requests
import pandas as pd
from datetime import datetime

url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

all_records = []
start_year = datetime.now().year - 5   # last 5 years
end_year = datetime.now().year

for year in range(start_year, end_year + 1):
    for month in range(1, 13):
        start_date = f"{year}-{month:02d}-01"
        if month == 12:
            end_date = f"{year+1}-01-01"
        else:
            end_date = f"{year}-{month+1:02d}-01"

        params = {
            "format": "geojson",
            "starttime": start_date,
            "endtime": end_date,
            "minmagnitude": 3
        }

        response = requests.get(url, params=params)
        if response.status_code != 200:
            print(f"⚠️ Failed for {start_date}: {response.text[:200]}")
            continue

        try:
            data = response.json()
        except Exception as e:
            print(f"⚠️ JSON error for {start_date}: {e}")
            continue

        for f in data["features"]:
            p = f["properties"]
            g = f["geometry"]["coordinates"]
            all_records.append({
                "id": f.get("id"),
                "time": pd.to_datetime(p.get("time"), unit="ms"),
                "updated": pd.to_datetime(p.get("updated"), unit="ms"),
                "latitude": g[1] if g else None,
                "longitude": g[0] if g else None,
                "depth_km": g[2] if g else None,
                "mag": p.get("mag")

            })

df = pd.DataFrame(all_records)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print(df.head())


Rows: 123136
Columns: 26
             id                    time                 updated  latitude  \
0  pr2020031037 2020-01-31 23:42:29.870 2020-04-18 22:10:07.040   18.8990   
1    us60007lub 2020-01-31 23:32:51.609 2020-04-18 22:10:07.040   38.4961   
2    us60007lul 2020-01-31 23:06:11.255 2020-04-18 22:10:07.040  -12.2747   
3    us60007lsc 2020-01-31 22:10:55.844 2020-04-18 22:10:06.040   -6.4065   
4    us60007lq0 2020-01-31 21:05:32.436 2020-04-18 22:10:06.040   41.4433   

   longitude  depth_km  mag magType  \
0   -67.8238     10.00  3.2      md   
1    39.3382     10.00  4.7     mwr   
2   -76.6148     75.51  4.7      mb   
3   129.1523    221.39  5.0     mww   
4    19.4321     10.00  4.4      mb   

                                         place    status  ...  net  \
0  70 km ENE of Punta Cana, Dominican Republic  reviewed  ...   pr   
1                  6 km NNE of Sivrice, Turkey  reviewed  ...   us   
2               21 km ENE of San Bartolo, Peru  reviewed  ...   us 

In [ ]:
df.columns

Index(['id', 'time', 'updated', 'latitude', 'longitude', 'depth_km', 'mag',
       'magType', 'place', 'status', 'tsunami', 'alert', 'felt', 'cdi', 'mmi',
       'sig', 'net', 'code', 'ids', 'sources', 'types', 'nst', 'dmin', 'rms',
       'gap', 'type'],
      dtype='object')

In [ ]:
print(df)

                  id                    time                 updated  \
0       pr2020031037 2020-01-31 23:42:29.870 2020-04-18 22:10:07.040   
1         us60007lub 2020-01-31 23:32:51.609 2020-04-18 22:10:07.040   
2         us60007lul 2020-01-31 23:06:11.255 2020-04-18 22:10:07.040   
3         us60007lsc 2020-01-31 22:10:55.844 2020-04-18 22:10:06.040   
4         us60007lq0 2020-01-31 21:05:32.436 2020-04-18 22:10:06.040   
...              ...                     ...                     ...   
118369    us7000qsy1 2025-09-01 03:18:12.227 2025-09-01 04:19:29.040   
118370    us7000qsxz 2025-09-01 02:59:51.485 2025-09-01 04:12:41.040   
118371    us7000qsxx 2025-09-01 02:43:45.408 2025-09-01 03:00:30.040   
118372    us7000qsxt 2025-09-01 01:59:21.447 2025-09-01 02:16:15.040   
118373    nn00903589 2025-09-01 01:18:08.850 2025-09-02 01:32:19.780   

        latitude  longitude  depth_km  mag magType  \
0        18.8990   -67.8238    10.000  3.2      md   
1        38.4961    39.3382

✅ What You Have Now

Core fields: time, latitude, longitude, depth_km, mag, magType, place

Seismic metadata: status, tsunami, alert, felt, cdi, mmi

Network details: net, code, ids, sources, types

Station quality metrics: nst, dmin, rms, gap

Administrative info: id, updated, type

In [ ]:
✅ Summary Table

| Parameter         | Practical Limit                    | Recommendation                    |
| ----------------- | ---------------------------------- | --------------------------------- |
| Records per query | ~20,000                            | Split by date or region           |
| Request size      | ~50–100 MB                         | Use compressed CSV/JSON           |
| Pagination        | Supported via `limit` and `offset` | Use for smaller incremental pulls |
| Best practice     | Monthly or regional slices         | Combine results in Pandas         |
